# Qwen3.6 を Google Colab (A100) で動かす

- 推論エンジン: **vLLM**（OpenAI 互換 API サーバをバックグラウンドで起動し、Python から呼ぶ）
- モデル: `Qwen/Qwen3.6-35B-A3B`（MoE, 総 35B / 活性 3B, 高速）を既定に、GPU メモリから自動選択
  - A100 **80GB** → 公式 BF16 版 `Qwen/Qwen3.6-35B-A3B`（約 67GB）
  - A100 **40GB** → 4bit 量子化版 `QuantTrio/Qwen3.6-35B-A3B-AWQ`（約 24GB, コミュニティ製）

## 使い方
1. メニュー「ランタイム」→「ランタイムのタイプを変更」→ **A100 GPU**（可能なら「ハイメモリ」も ON）
2. 上から順にセルを実行（初回はモデルのダウンロードに 5〜15 分程度かかります）
3. （任意）左の🔑「シークレット」に `HF_TOKEN` を登録するとダウンロードが安定します

In [ ]:
# GPU 確認
import subprocess

out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
).stdout.strip().splitlines()[0]
GPU_NAME, GPU_MEM_MIB = [s.strip() for s in out.split(",")]
GPU_MEM_GB = int(GPU_MEM_MIB) / 1024
print(f"GPU: {GPU_NAME}  /  VRAM: {GPU_MEM_GB:.0f} GB")
assert "A100" in GPU_NAME, "A100 ランタイムになっていません。ランタイムのタイプを変更してください。"

In [ ]:
# インストール（vLLM は別プロセスで動かすので、通常ランタイム再起動は不要）
!uv pip install --system -q -U vllm --torch-backend=auto
!uv pip install --system -q -U openai
!vllm --version

In [ ]:
# ===== 設定 =====
import os, secrets

MODEL_CHOICE = "auto"   # "auto" / "35b-bf16" / "27b-bf16" / "35b-awq"
MAX_MODEL_LEN = 32768   # 最大コンテキスト長（OOM になったら 16384 などに下げる）
USE_VISION = False      # 画像入力も使うなら True（メモリ消費が増える）

MODELS = {
    "35b-bf16": "Qwen/Qwen3.6-35B-A3B",           # 80GB 向け（公式）
    "27b-bf16": "Qwen/Qwen3.6-27B",               # 80GB 向け（公式, dense で遅め）
    "35b-awq":  "QuantTrio/Qwen3.6-35B-A3B-AWQ",  # 40GB 向け（コミュニティ製 4bit）
}
if MODEL_CHOICE == "auto":
    MODEL_CHOICE = "35b-bf16" if GPU_MEM_GB > 70 else "35b-awq"
MODEL_ID = MODELS[MODEL_CHOICE]
SERVED_NAME = "qwen3.6"
PORT = 8000
GPU_UTIL = 0.94 if MODEL_CHOICE.endswith("bf16") else 0.90
# 外部公開したときに第三者に使われないよう、API にはキーを掛ける
API_KEY = "sk-" + secrets.token_urlsafe(24)

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN を読み込みました")
except Exception:
    print("HF_TOKEN なしで続行します（公開モデルなので動作はします）")

print(f"使用モデル: {MODEL_ID}  (max_model_len={MAX_MODEL_LEN}, gpu_util={GPU_UTIL})")

In [ ]:
# vLLM サーバをバックグラウンドで起動
import subprocess

cmd = [
    "vllm", "serve", MODEL_ID,
    "--served-model-name", SERVED_NAME,
    "--port", str(PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    # GatedDeltaNet 層は chunked prefill で分割されると出力が崩れることがあるため、コンテキスト長と揃える
    "--max-num-batched-tokens", str(MAX_MODEL_LEN),
    "--max-num-seqs", "8",
    "--gpu-memory-utilization", str(GPU_UTIL),
    "--reasoning-parser", "qwen3",
    "--trust-remote-code",
    "--api-key", API_KEY,
]
if not USE_VISION:
    cmd.append("--language-model-only")

log_file = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
print("起動コマンド:", " ".join(cmd).replace(API_KEY, "***"))
print("ログ: vllm.log")

In [ ]:
# 起動完了まで待つ（ダウンロード込みで最大 40 分）
import time, urllib.request

def tail(path="vllm.log", n=15):
    with open(path, errors="ignore") as f:
        return "".join(f.readlines()[-n:])

t0 = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM サーバが終了しました。ログ末尾:\n" + tail(n=40))
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2)
        break
    except Exception:
        pass
    if time.time() - t0 > 40 * 60:
        raise TimeoutError("40 分以内に起動しませんでした。ログ末尾:\n" + tail(n=40))
    time.sleep(20)
    print(f"--- {time.time() - t0:.0f} s 経過 ---\n{tail(n=3)}")

print(f"準備完了（{time.time() - t0:.0f} s）")

In [ ]:
# チャット関数（ストリーミング表示）
from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key=API_KEY)

# 公式推奨のサンプリング設定
SAMPLING = {
    "thinking": dict(temperature=1.0, top_p=0.95, presence_penalty=1.5),
    "coding":   dict(temperature=0.6, top_p=0.95, presence_penalty=0.0),
    "instruct": dict(temperature=0.7, top_p=0.80, presence_penalty=1.5),
}


def chat(messages, mode="thinking", max_tokens=None, show_thinking=True):
    """mode: "thinking"（既定）/ "coding" / "instruct"（思考なしで即答）"""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    extra = {"top_k": 20}
    if mode == "instruct":
        extra["chat_template_kwargs"] = {"enable_thinking": False}
    if max_tokens is None:
        max_tokens = 4096 if mode == "instruct" else 16384

    stream = client.chat.completions.create(
        model=SERVED_NAME, messages=messages, max_tokens=max_tokens,
        stream=True, extra_body=extra, **SAMPLING[mode],
    )
    answer, state = [], None
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        reasoning = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
        if reasoning and show_thinking:
            if state != "think":
                print("\n[思考]\n", end="")
                state = "think"
            print(reasoning, end="", flush=True)
        if delta.content:
            if state != "answer":
                print("\n\n[回答]\n", end="")
                state = "answer"
            print(delta.content, end="", flush=True)
            answer.append(delta.content)
    print()
    return "".join(answer)

In [ ]:
# 例1: 思考モード（既定）
_ = chat("宇宙線ミューオンが地下で減衰する理由を、高校生にわかるように3行で説明して。")

In [ ]:
# 例2: 思考なし（即答）モード
_ = chat("PHITS とは何ですか？一文で。", mode="instruct")

In [ ]:
# 例3: 会話履歴を保つ簡易チャット（履歴には回答部分だけを残す）
class Conversation:
    def __init__(self, system=None, mode="thinking"):
        self.mode = mode
        self.messages = [{"role": "system", "content": system}] if system else []

    def __call__(self, text, **kw):
        self.messages.append({"role": "user", "content": text})
        reply = chat(self.messages, mode=kw.pop("mode", self.mode), **kw)
        self.messages.append({"role": "assistant", "content": reply})
        return reply


conv = Conversation(system="あなたは素粒子・原子核物理に詳しいアシスタントです。日本語で答えてください。")
_ = conv("熱中性子と速中性子の違いは？")
_ = conv("それぞれを測る代表的な検出器は？")

## 手元の PC から ChatGPT のように使う

どちらか好きな方法を選んで実行します（両方同時でも可）。どちらも **Colab のタブを開いている間だけ** 使えます。

- **方法A（おすすめ・インストール不要）**: Gradio のチャット画面を公開し、表示された URL を PC のブラウザで開く
- **方法B**: OpenAI 互換 API を Cloudflare トンネルで公開し、PC のチャットアプリ（Open WebUI, Chatbox など）や Python から接続

In [ ]:
# 方法A: Gradio のチャット画面を公開（表示される https://xxxx.gradio.live を PC のブラウザで開く）
import re, secrets, subprocess, sys

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

UI_USER = "kek"
UI_PASSWORD = secrets.token_urlsafe(8)
THINK_BLOCK = re.compile(r"<details>.*?</details>\s*", re.S)


def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, (list, tuple)):
        return "".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return "" if content is None else str(content)


def _history_to_messages(history):
    # Gradio のバージョンによって履歴の形式が違うので両方に対応し、思考過程は履歴から除く
    msgs = []
    for item in history:
        pairs = [(item.get("role"), item.get("content"))] if isinstance(item, dict) else \
                [("user", item[0]), ("assistant", item[1])]
        for role, content in pairs:
            text = THINK_BLOCK.sub("", _text(content)).strip()
            if role in ("user", "assistant") and text:
                msgs.append({"role": role, "content": text})
    return msgs


def respond(message, history, mode, system_prompt):
    msgs = [{"role": "system", "content": system_prompt}] if system_prompt.strip() else []
    msgs += _history_to_messages(history)
    msgs.append({"role": "user", "content": _text(message)})

    extra = {"top_k": 20}
    if mode == "instruct":
        extra["chat_template_kwargs"] = {"enable_thinking": False}
    stream = client.chat.completions.create(
        model=SERVED_NAME, messages=msgs, max_tokens=4096 if mode == "instruct" else 16384,
        stream=True, extra_body=extra, **SAMPLING[mode],
    )
    thinking, answer = "", ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        thinking += getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None) or ""
        answer += delta.content or ""
        shown = ""
        if thinking:
            label = "思考過程" if answer else "考え中…"
            shown = f"<details><summary>{label}</summary>\n\n{thinking}\n\n</details>\n\n"
        yield shown + answer


demo = gr.ChatInterface(
    respond,
    title="Qwen3.6 (Colab A100)",
    additional_inputs=[
        gr.Dropdown(
            [("思考あり", "thinking"), ("コーディング", "coding"), ("即答（思考なし）", "instruct")],
            value="thinking", label="モード",
        ),
        gr.Textbox(value="日本語で答えてください。", label="システムプロンプト"),
    ],
)
demo.queue().launch(share=True, auth=(UI_USER, UI_PASSWORD), prevent_thread_lock=True)
print(f"\nログイン → ユーザー名: {UI_USER}  /  パスワード: {UI_PASSWORD}")
print("上に表示された public URL (https://....gradio.live) を PC のブラウザで開いてください")

In [ ]:
# 方法B: OpenAI 互換 API を Cloudflare クイックトンネルで公開（アカウント不要、URL は起動ごとに変わる）
import os, re, subprocess, time

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod("cloudflared", 0o755)

tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0)
        break
if PUBLIC_URL is None:
    raise RuntimeError("トンネル URL を取得できませんでした。cloudflared.log を確認してください。")

print("PC 側のアプリに設定する値（アクセス可能になるまで 10〜30 秒ほどかかることがあります）")
print(f"  Base URL : {PUBLIC_URL}/v1")
print(f"  API Key  : {API_KEY}")
print(f"  Model    : {SERVED_NAME}")

### 方法B: PC 側の設定例

上のセルで表示された `Base URL` / `API Key` / `Model` を使います。

**Open WebUI（ChatGPT に最も近い画面。Docker が必要）**
```bash
docker run -d -p 3000:8080 \
  -e OPENAI_API_BASE_URL="<Base URL>" -e OPENAI_API_KEY="<API Key>" \
  -v open-webui:/app/backend/data --name open-webui ghcr.io/open-webui/open-webui:main
```
→ ブラウザで http://localhost:3000 を開く。トンネル URL が変わったら「管理者設定 → 接続」で書き換える

**Chatbox などのデスクトップアプリ**: プロバイダに「OpenAI API 互換」を選び、上の 3 つの値を入力

**Python**
```python
from openai import OpenAI
client = OpenAI(base_url="<Base URL>", api_key="<API Key>")
r = client.chat.completions.create(model="qwen3.6", messages=[{"role": "user", "content": "こんにちは"}])
print(r.choices[0].message.content)
```

## トラブルシューティング

- **起動中に `CUDA out of memory`**: 設定セルで `MAX_MODEL_LEN` を 16384 に下げる。80GB で BF16 が厳しい場合は `MODEL_CHOICE = "35b-awq"` に切り替える
- **40GB で AWQ 版が起動しない**: vLLM のバージョンとの相性が原因のことがあります。`vllm.log` を確認し、`palmfuture/Qwen3.6-35B-A3B-GPTQ-Int4` など他の 4bit 版を `MODELS` に追加して試す
- **`--kv-cache-dtype fp8` は付けない**: Qwen3.6 のハイブリッド構造（GatedDeltaNet）で出力が壊れる既知の問題があります
- **設定を変えて再起動したい**: 下のセルでサーバを停止 → 設定セルから再実行
- **ランタイムが切断された**: モデルは再ダウンロードになります（Colab のローカルディスクは消えるため）

In [ ]:
# サーバ停止（GPU メモリ解放）。公開中のチャット画面・トンネルも閉じる
if "demo" in globals():
    demo.close()
if "tunnel" in globals():
    tunnel.terminate()
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()
log_file.close()
print("停止しました")